# 10.5 Reading and writing the same table

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

To read `data` from a relational `table` and then write results to the same `table`, you can
use a pair of `table` declarations that reference the same file and `table` names. You may
also be able to combine these declarations into one that specifies some columns to be read
and others to be written. This section gives examples and instructions for both of these
possibilities.

Reading and writing using two `table` declarations
A single external `table` can be read by one `table` declaration and later written by
another. The two `table` declarations follow the rules for reading and writing given
above.

In this situation, however, one usually wants `write table` to add or rewrite
selected columns, rather than overwriting the entire `table`. This preference can be communicated
to the AMPL `table` handler by including input as well as output columns in the
`table` declaration that is to be used for writing. Columns intended for input to AMPL
can be distinguished from those intended for output to the external `table` by specifying a
read/write status column by column (rather than for the `table` as a whole).

As an example, an external `table` for diet.mod might consist of columns cost,
f_min and f_max containing input for the `model`, and a column Buy containing the
results. If this is maintained as a Microsoft Access `table` named Diet within a file
diet.mdb, the `table` declaration for reading `data` into AMPL could be

```ampl
table FoodInput IN "ODBC" "diet1.mdb" "Diet":
   FOOD <- [FoodName], cost, f_min, f_max;
```

The corresponding declaration for writing the results would have a different AMPL tablename
but would refer to the same Access `table` and file:

```ampl
table FoodOutput "ODBC" "diet1.mdb" "Diet":
   [FoodName], cost IN, f_min IN, Buy OUT, f_max IN;
```

When `read table` FoodInput is executed, only the three columns listed in the
`table` FoodInput declaration are read; if there is an existing column named Buy, it is
ignored. Later, when the problem has been solved and `write table` FoodOutput is
executed, only the one column that has read/write status OUT in the `table` FoodOutput
declaration is written to the Access `table`, while the table's other columns are left
unmodified.

Although details may vary with the database software used, the general convention is
that overwriting of an entire existing `table` or file is intended only when all `data` columns
in the `table` declaration have read/write status OUT. Selective rewriting or addition of
columns is intended otherwise. Thus if our AMPL `table` for output had been declared

```ampl
table FoodOutput "ODBC" "diet1.mdb" "Diet":
                [FoodName], Buy OUT;
```

then all of the `data` columns in Access `table` Diet would have been deleted by write
`table` FoodOutput, but the alternative

```ampl
table FoodOutput "ODBC" "diet1.mdb" "Diet":
                [FoodName], Buy;
```

would have only overwritten the column Buy, as in the example we originally gave, since
there is a `data` column (namely Buy itself) that does not have read/write status OUT. (The
default, when no status is given, is INOUT.)

Reading and writing using the same `table` declaration
In many cases, all of the information for both reading and writing an external `table`
can be specified in the same `table` declaration. The key-spec may use the arrow <- to
read contents of the key columns into an AMPL set, -> to write members of an AMPL set
into the key columns, or <-> to do both. A `data`-spec may specify read/write status IN
for a column that will only be read into AMPL, OUT for a column that will only be written
out from AMPL, or INOUT for a column that will be both read and written.

A `read table` `table`-name command reads only the key or `data` columns that are
specified in the declaration of `table`-name as being IN or INOUT. A `write table`
`table`-name command analogously writes to only the columns that are specified as OUT or
INOUT.

As an example, the declarations defining FoodInput and FoodOutput above
could be replaced by

```ampl
table Foods "ODBC" "diet1.mdb" "Diet":
   FOOD <- [FoodName], cost IN, f_min IN, Buy OUT, f_max IN;
```

A `read table` Foods would then read only from key column FoodName and `data`
columns cost, f_min and f_max. A later `write table` Foods would write only to
the column Buy.